In [0]:
%sql
create table batch4.class.src_emp(
    ID INT,
    Name String,
    Salary INT,
    Location String,
    DeptID INT,
    LastUpdateDate TIMESTAMP
)
USING DELTA

In [0]:
%sql
insert into batch4.class.src_emp values 
(1,'John',5000,'New York',1,current_timestamp()),
(2,'Mary',6000,'New Jersey',2,current_timestamp()),
(3,'Mike',7000,'California',1,current_timestamp())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
Update batch4.class.src_emp
set Location = 'Houston',
    LastUpdateDate = current_timestamp()
where ID = 1

num_affected_rows
1


In [0]:
%sql
insert into batch4.class.src_emp values 
(4,'Raj',5000,'Mumbai',1,current_timestamp()),
(5,'Ram',6000,'Delhi',2,current_timestamp()),
(6,'Ravi',7000,'Hyderabad',1,current_timestamp())

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from batch4.class.src_emp

ID,Name,Salary,Location,DeptID,LastUpdateDate
1,John,5000,New York,1,2026-10-08T04:48:52.165Z
2,Mary,6000,New Jersey,2,2026-10-08T04:48:52.165Z
3,Mike,7000,California,1,2026-10-08T04:48:52.165Z


In [0]:
%sql
create table batch4.class.dim_emp_scd2 (
    ID INT,
    Name String,
    Salary INT,
    Location String,
    DeptID INT,
    Hashbyte STRING,
    StartDate TIMESTAMP,
    EndDate timestamp,
    IsCurrent INT
)

In [0]:
from pyspark.sql.functions import sha2, concat_ws, current_timestamp

df = spark.sql("select * from batch4.class.src_emp")

trans_df = df.withColumn("Hashbyte", sha2(concat_ws(':', *df.columns), 256)) \
             .drop("LastUpdateDate")

display(trans_df)

ID,Name,Salary,Location,DeptID,Hashbyte
1,John,5000,Houston,1,9095d4b5d09454bd6d844a5d3f8e56f83da4d32303e5f8ea029642ff03146f35
2,Mary,6000,New Jersey,2,e05c2a8537893b3b03eeba4d2adecd9055f109ddecda47ad2415245d3aa2841d
3,Mike,7000,California,1,1611de020d3ba0710d288d8f583cdc95e5b839541a3a19529f2b0bfbe8030ad9
4,Raj,5000,Mumbai,1,80172a740538dcd234d7cca708cd6be78c03c8baf11c77215acad58c55d63c61
5,Ram,6000,Delhi,2,058c432e871dd70b235b6039762ce77fcf2a8d84d738415336e45b45f69eb2cc
6,Ravi,7000,Hyderabad,1,c1ef4f510a63038d4dca105baf0a9129b22dc19a86bb928c795cbb5fe1f954b3


# SCD Type2 Logic

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

target = DeltaTable.forName(spark, "batch4.class.dim_emp_scd2")

target.alias("t") \
    .merge(trans_df.alias("s"), "t.ID = s.ID") \
    .whenMatchedUpdate(condition = "t.Hashbyte != s.Hashbyte AND t.IsCurrent = 1", set = {"IsCurrent": "0", "EndDate": current_timestamp()}) \
    .execute()


target.alias("t") \
    .merge(trans_df.alias("s"), "t.ID = s.ID AND IsCurrent = 1") \
    .whenNotMatchedInsert(values = {"ID": "s.ID", "Name": "s.Name", "Salary": "s.Salary", "Location": "s.Location", "DeptID": "s.DeptID", "Hashbyte": "s.Hashbyte", "StartDate": current_timestamp(), "EndDate": "to_date('9999-12-31')", "IsCurrent": "1"}) \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from batch4.class.dim_emp_scd2

ID,Name,Salary,Location,DeptID,Hashbyte,StartDate,EndDate,IsCurrent
2,Mary,6000,New Jersey,2,e05c2a8537893b3b03eeba4d2adecd9055f109ddecda47ad2415245d3aa2841d,2026-10-08T05:06:45.403Z,9999-12-31T00:00:00.000Z,1
3,Mike,7000,California,1,1611de020d3ba0710d288d8f583cdc95e5b839541a3a19529f2b0bfbe8030ad9,2026-10-08T05:06:45.403Z,9999-12-31T00:00:00.000Z,1
1,John,5000,New York,1,4690a268fd4056e60431b4359853a837f683c7c7152a682152d403a5f8a8ae56,2026-10-08T05:06:45.403Z,2026-10-08T05:08:06.981Z,0
4,Raj,5000,Mumbai,1,80172a740538dcd234d7cca708cd6be78c03c8baf11c77215acad58c55d63c61,2026-10-08T05:08:11.920Z,9999-12-31T00:00:00.000Z,1
5,Ram,6000,Delhi,2,058c432e871dd70b235b6039762ce77fcf2a8d84d738415336e45b45f69eb2cc,2026-10-08T05:08:11.920Z,9999-12-31T00:00:00.000Z,1
6,Ravi,7000,Hyderabad,1,c1ef4f510a63038d4dca105baf0a9129b22dc19a86bb928c795cbb5fe1f954b3,2026-10-08T05:08:11.920Z,9999-12-31T00:00:00.000Z,1
1,John,5000,Texas,1,9095d4b5d09454bd6d844a5d3f8e56f83da4d32303e5f8ea029642ff03146f35,2026-10-08T05:08:11.920Z,2026-10-08T05:11:21.573Z,0
1,John,5000,Houston,1,9095d4b5d09454bd6d844a5d3f8e56f83da4d32303e5f8ea029642ff03146f35,2026-10-08T05:11:26.805Z,9999-12-31T00:00:00.000Z,1
